In [ ]:
import osimport sysimport yamlimport randomimport numpy as npimport torchimport torch.nn as nnfrom typing import Optionalimport loggingfrom logging import Logger### metricsdef MAE(pred, true):    return torch.abs(pred - true)def MSE(pred, true):    return (pred - true) ** 2def MAPE(pred, true):    return torch.abs((pred - true) / true)def SMAPE(pred, true):    # Avoid division by zero by adding a small constant    denominator = (torch.abs(true) + torch.abs(pred)) / 2 + 1e-8    # Calculate the SMAPE    smape_value = torch.mean(torch.abs(pred - true) / denominator)    return smape_valuedef masked_loss(y_pred, y_true, loss_func):    y_true[y_true < 1e-4] = 0    mask = (y_true != 0).float()    mask /= mask.mean()  # assign the sample weights of zeros to nonzero-values    loss = loss_func(y_pred, y_true)    loss = loss * mask    loss[loss != loss] = 0    return loss.mean()def masked_rmse_loss(y_pred, y_true):    y_true[y_true < 1e-4] = 0    mask = (y_true != 0).float()    mask /= mask.mean()    loss = torch.pow(y_pred - y_true, 2)    loss = loss * mask    loss[loss != loss] = 0    return torch.sqrt(loss.mean())def compute_all_metrics(y_pred, y_true):    mae = masked_loss(y_pred, y_true, MAE).item()    rmse = masked_rmse_loss(y_pred, y_true).item()    smape = masked_loss(y_pred, y_true, SMAPE).item()    return mae, smape, rmse### toolsdef count_parameters(model):    return sum(p.numel() for p in model.parameters() if p.requires_grad)class EarlyStopping:    def __init__(self, patience=7, verbose=False, delta=0, logger: Optional[Logger]=None):        self.patience = patience        self.verbose = verbose        self.counter = 0        self.best_score = None        self.early_stop = False        self.val_loss_min = np.Inf        self.delta = delta        self.logger = logger    def __call__(self, val_loss, model, path):        score = -val_loss        if self.best_score is None:            self.best_score = score            self.save_checkpoint(val_loss, model, path)        elif score < self.best_score + self.delta:            self.counter += 1            message = f'EarlyStopping counter: {self.counter} out of {self.patience}'            self.logger.info(message)            if self.counter >= self.patience:                self.early_stop = True        else:            self.best_score = score            self.save_checkpoint(val_loss, model, path)            self.counter = 0    def save_checkpoint(self, val_loss, model, path):        if self.verbose:            message = f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...'            self.logger.info(message)        torch.save(model.state_dict(), path + '/' + 'checkpoint.pth')        self.val_loss_min = val_lossdef load_config(config_path):    with open(config_path, 'r') as file:        config = yaml.safe_load(file)    return configdef parsing_syntax(unknown):    unknown_dict = {}    key = None    for arg in unknown:        if arg.startswith('--'):            key = arg.lstrip('--')            unknown_dict[key] = None        else:            if key:                unknown_dict[key] = arg                key = None    return unknown_dictclass ConfigDict(dict):    def __init__(self, *args, **kwargs):        super(ConfigDict, self).__init__(*args, **kwargs)        for key, value in self.items():            if isinstance(value, dict):                self[key] = ConfigDict(value)            if key == 'data' and isinstance(value, str):                dataset_config = load_config("../Model_Config/dataset_config/{}".format(value + ".yaml"))                self[key]= ConfigDict(dataset_config)    def __getattr__(self, item):        try:            return self[item]        except KeyError:            raise AttributeError(f"'ConfigDict' object has no attribute '{item}'")    def __setattr__(self, key, value):        self[key] = valuedef update_config(config, unknown_args):    for key, value in unknown_args.items():        config_path = key.split('-')        cur = config        for node in config_path:            assert node in cur.keys(), "path not exist"            if isinstance(cur[node], ConfigDict):                cur = cur[node]            else:                try:                    cur[node] = eval(value)                except NameError:                    cur[node] = value    return configdef load_graph_data(dataset_path):    npz_path = os.path.join(dataset_path, 'graph_data.npz')    data = np.load(npz_path)    adj_mx = data['adj_mx']    edge_index = data['edge_index']    edge_attr = data['edge_attr']   # {diff_dist, dist_km, direction}    node_attr = data['node_attr']    return adj_mx, edge_index.T, edge_attr, node_attrdef fix_seed(seed):    os.environ['PYTHONHASHSEED'] = str(seed)    random.seed(seed)    np.random.seed(seed)    torch.manual_seed(seed)    torch.cuda.manual_seed(seed)    torch.cuda.manual_seed_all(seed)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark = Falsedef get_logger(log_dir, name, log_filename='info.log', level=logging.INFO, to_stdout=True):    logger = logging.getLogger(name)    logger.setLevel(level)    # Add console handler.    if to_stdout:        console_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')        console_handler = logging.StreamHandler(sys.stdout)        console_handler.setFormatter(console_formatter)        logger.addHandler(console_handler)    # Add file handler and stdout handler    if log_dir:        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%m-%d %H:%M')        file_handler = logging.FileHandler(os.path.join(log_dir, log_filename))        file_handler.setFormatter(formatter)        logger.addHandler(file_handler)        logger.info('Log directory: %s', log_dir)    return loggerdef init_network_weights(net, std=0.1):    """    Just for nn.Linear net.    """    for m in net.modules():        if isinstance(m, nn.Linear):            nn.init.normal_(m.weight, mean=0, std=std)            nn.init.constant_(m.bias, val=0)def split_last_dim(data):    last_dim = data.size()[-1]    last_dim = last_dim // 2    res = data[..., :last_dim], data[..., last_dim:]    return resdef exchange_df_column(df, col1, col2):    """    exchange df column    :return new_df    """    assert (col1 in df.columns) and (col2 in df.columns)    df[col1], df[col2] = df[col2].copy(), df[col1].copy()    df = df.rename(columns={col1: 'temp', col2: col1})    df = df.rename(columns={'temp': col2})    return dfimport osimport numpy as npimport pandas as pdfrom metpy.units import unitsimport metpy.calc as mpcalcfrom typing import Union, Listfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.preprocessing import StandardScalerimport chinese_calendar as calendarclass Dataset_KnowAir(Dataset):    def __init__(self, root_path, flag='train', seq_len=24, pred_len=24,                 freq='3h', scale=True, embed=0,                 normalized_col: Union[str, List[int]]='default'):        if normalized_col == 'default':            self.normalized_col = np.arange(0, 6)        else:            self.normalized_col = normalized_col        self.seq_len = seq_len        self.pred_len = pred_len        self.window_size = seq_len + pred_len        self.scale = scale        self.embed = embed        if scale:            self.scaler = StandardScaler()        else:            self.scaler = None        assert flag in ['train', 'test', 'val']        type_map = {'train': 0, 'val': 1, 'test': 2}        self.set_type = type_map[flag]        self.root_path = root_path        self.station_info = pd.read_csv(os.path.join(self.root_path, "station.csv"))        self.stations_npy = os.path.join(self.root_path, "KnowAir.npy")        metero_var = ['100m_u_component_of_wind', '100m_v_component_of_wind', '2m_dewpoint_temperature',                       '2m_temperature', 'boundary_layer_height', 'k_index', 'relative_humidity+950',                       'relative_humidity+975', 'specific_humidity+950', 'surface_pressure',                       'temperature+925', 'temperature+950', 'total_precipitation', 'u_component_of_wind+950',                       'v_component_of_wind+950', 'vertical_velocity+950', 'vorticity+950']        metero_use = ['2m_temperature', 'surface_pressure', 'relative_humidity+950',                      '100m_u_component_of_wind', '100m_v_component_of_wind']        self.metero_idx = [metero_var.index(var) for var in metero_use]        self.time_idx = pd.date_range(start='2015-01-01', end='2018-12-31 21:00', freq='3H')        self.__process_raw_data__()        self.__read_data__()    def __process_raw_data__(self):        raw_data = np.load(self.stations_npy)        self.pm25 = raw_data[:, :, -1:]        self.feature = raw_data[:, :, :-1]        self.feature = self.feature[:, :, self.metero_idx]        u = self.feature[:, :, -2] * units.meter / units.second   # m/s        v = self.feature[:, :, -1] * units.meter / units.second   # m/s        speed = 3.6 * mpcalc.wind_speed(u, v)._magnitude    # km/h        direc = mpcalc.wind_direction(u, v)._magnitude        self.feature[:, :, -2] = speed        self.feature[:, :, -1] = direc        self.raw_data = np.concatenate([self.pm25, self.feature], axis=-1)  # T x N x D    def __read_data__(self):        # 2:1:1        border1s = [0, int(len(self.raw_data) * 0.5), int(len(self.raw_data) * 0.75)]        border2s = [int(len(self.raw_data) * 0.5), int(len(self.raw_data) * 0.75), len(self.raw_data)]        self.train_border = (border1s[0], border2s[0])        border1 = border1s[self.set_type]        border2 = border2s[self.set_type]        self.data = self.raw_data[border1: border2]        if self.embed:            self.time_info = self.cal_time_info(self.time_idx[border1: border2]).values        if self.scale:            train_set = self.raw_data[self.train_border[0]: self.train_border[1], :, :]            T, N, D = self.data.shape            self.scaler.fit(train_set.reshape(-1, D)[:, self.normalized_col])            self.data = self.data.reshape(-1, D)            self.data[:, self.normalized_col] = self.scaler.transform(self.data[:, self.normalized_col])            self.data = self.data.reshape(T, N, D)    def cal_time_info(self, time_idx):        def check_holiday(date):            return 1 if calendar.is_holiday(date) or calendar.is_in_lieu(date) else 0        time_info = pd.DataFrame({            'time': time_idx,            'hour_of_day': time_idx.hour,  # hour-day            'day_of_week': time_idx.dayofweek,  # day-week            'day_of_month': time_idx.day - 1,  # day-month            'month_of_year': time_idx.month - 1,  # month-year        })        time_info['is_holiday'] = [check_holiday(d.date()) for d in time_idx]        time_info.set_index('time', inplace=True)        return time_info    def __len__(self):        return len(self.data) - self.window_size + 1    def __getitem__(self, idx):        x_start = idx        x_end = x_start + self.seq_len        y_start = idx + self.seq_len        y_end = y_start + self.pred_len        seq_x = self.data[x_start: x_end]        seq_y = self.data[y_start: y_end]        if self.embed:            seq_x_time_info = self.time_info[x_start: x_end]            seq_x_time_info = np.expand_dims(seq_x_time_info, axis=1).repeat(seq_x.shape[1],axis=1)            seq_x = np.concatenate([seq_x, seq_x_time_info], axis=2)            seq_y_time_info = self.time_info[y_start: y_end]            seq_y_time_info = np.expand_dims(seq_y_time_info, axis=1).repeat(seq_x.shape[1],axis=1)            seq_y = np.concatenate([seq_y, seq_y_time_info], axis=2)        return seq_x, seq_y    def inverse_transform(self, data):        assert self.scale is True        pm25_mean = self.scaler.mean_[0]        pm25_std = self.scaler.scale_[0]        return (data * pm25_std) + pm25_meandef data_provider(args, flag):    data_args = args.data    model_args = args.model    Data = data_dict[data_args.data_name]    if flag == 'train':        shuffle_flag = True        drop_last = True    else:        shuffle_flag = False        drop_last = False    batch_size = data_args.batch_size    if data_args.data_name == "Beijing1718_old":        data_set = Data(            root_path=data_args.root_path,            flag=flag        )    else:        data_set = Data(            root_path=data_args.root_path,            flag=flag,            seq_len=model_args.seq_len,            pred_len=model_args.horizon,            freq=data_args.interval,            embed=data_args.embed,            scale=True,            normalized_col=data_args.normalized_columns        )    print(flag, len(data_set))    data_loader = DataLoader(        data_set,        batch_size=batch_size,        shuffle=shuffle_flag,        num_workers=data_args.num_workers,        drop_last=drop_last)    return data_set, data_loaderdata_dict = {    'KnowAir': Dataset_KnowAir}import mathimport torchimport torch.nn.functional as Fimport torch.nn as nnfrom torch_geometric.nn import ChebConvfrom torch_geometric.utils import dense_to_sparsefrom torchdiffeq import odeint_adjoint as odeintclass AGCN(nn.Module):    def __init__(self, dim_in, dim_out, cheb_k):        super(AGCN, self).__init__()        self.cheb_k = cheb_k        self.weights = nn.Parameter(torch.FloatTensor(2*cheb_k*dim_in, dim_out)) # 2 is the length of support        self.bias = nn.Parameter(torch.FloatTensor(dim_out))        nn.init.xavier_normal_(self.weights)        nn.init.constant_(self.bias, val=0)            def forward(self, x, supports):        x_g = []                support_set = []        for support in supports:            support_ks = [torch.eye(support.shape[0]).to(support.device), support]            for k in range(2, self.cheb_k):                support_ks.append(torch.matmul(2 * support, support_ks[-1]) - support_ks[-2])             support_set.extend(support_ks)        for support in support_set:            x_g.append(torch.einsum("nm,bmc->bnc", support, x))        x_g = torch.cat(x_g, dim=-1) # B, N, 2 * cheb_k * dim_in        x_gconv = torch.einsum('bni,io->bno', x_g, self.weights) + self.bias  # b, N, dim_out        return x_gconv    class AGCRNCell(nn.Module):    def __init__(self, node_num, dim_in, dim_out, cheb_k):        super(AGCRNCell, self).__init__()        self.node_num = node_num        self.hidden_dim = dim_out        self.gate = AGCN(dim_in+self.hidden_dim, 2*dim_out, cheb_k)        self.update = AGCN(dim_in+self.hidden_dim, dim_out, cheb_k)    def forward(self, x, state, supports):        #x: B, num_nodes, input_dim        #state: B, num_nodes, hidden_dim        # print(x.shape, state.shape)        state = state.to(x.device)        input_and_state = torch.cat((x, state), dim=-1)        z_r = torch.sigmoid(self.gate(input_and_state, supports))        z, r = torch.split(z_r, self.hidden_dim, dim=-1)        candidate = torch.cat((x, z*state), dim=-1)        hc = torch.tanh(self.update(candidate, supports))        h = r*state + (1-r)*hc        return h    def init_hidden_state(self, batch_size):        return torch.zeros(batch_size, self.node_num, self.hidden_dim)    class STEncoder(nn.Module):    def __init__(self, node_num, dim_in, dim_out, cheb_k, num_layers):        super(STEncoder, self).__init__()        assert num_layers >= 1, 'At least one DCRNN layer in the Encoder.'        self.node_num = node_num        self.input_dim = dim_in        self.num_layers = num_layers        self.dcrnn_cells = nn.ModuleList()        self.dcrnn_cells.append(AGCRNCell(node_num, dim_in, dim_out, cheb_k))        for _ in range(1, num_layers):            self.dcrnn_cells.append(AGCRNCell(node_num, dim_out, dim_out, cheb_k))    def forward(self, x, init_state, supports):        # x: (B, T, N, D)        # init_state: (num_layers, B, N, hidden_dim)        assert x.shape[2] == self.node_num and x.shape[3] == self.input_dim        seq_length = x.shape[1]        current_inputs = x        output_hidden = []        for i in range(self.num_layers):            state = init_state[i]            inner_states = []            for t in range(seq_length):                state = self.dcrnn_cells[i](current_inputs[:, t, :, :], state, supports)                inner_states.append(state)            output_hidden.append(state)            current_inputs = torch.stack(inner_states, dim=1)        return current_inputs, output_hidden        def init_hidden(self, batch_size):        init_states = []        for i in range(self.num_layers):            init_states.append(self.dcrnn_cells[i].init_hidden_state(batch_size))        return init_statesclass STDecoder(nn.Module):    def __init__(self, node_num, dim_in, dim_out, cheb_k, num_layers):        super(STDecoder, self).__init__()        assert num_layers >= 1, 'At least one DCRNN layer in the Decoder.'        self.node_num = node_num        self.input_dim = dim_in        self.num_layers = num_layers        self.dcrnn_cells = nn.ModuleList()        self.dcrnn_cells.append(AGCRNCell(node_num, dim_in, dim_out, cheb_k))        for _ in range(1, num_layers):            self.dcrnn_cells.append(AGCRNCell(node_num, dim_out, dim_out, cheb_k))    def forward(self, xt, init_state, supports):        # xt: (B, N, D)        # init_state: (num_layers, B, N, hidden_dim)        assert xt.shape[1] == self.node_num and xt.shape[2] == self.input_dim        current_inputs = xt        output_hidden = []        for i in range(self.num_layers):            state = self.dcrnn_cells[i](current_inputs, init_state[i], supports)            output_hidden.append(state)            current_inputs = state        return current_inputs, output_hiddenclass LocalMemoryModule(nn.Module):    def __init__(self, num_nodes, d_model, tau=3, k_neighbors=8):        super().__init__()        self.num_nodes = num_nodes        self.d_model = d_model        self.tau = tau        self.k_neighbors = k_neighbors        self.q_proj = nn.Linear(d_model, d_model)        self.k_proj = nn.Linear(d_model, d_model)        self.v_proj = nn.Linear(d_model, d_model)        self.mlp = nn.Sequential(            nn.Linear(d_model, d_model),            nn.GELU(),            nn.Linear(d_model, d_model)        )    def forward(self, h_e, x_orig):        batch_size, seq_len, num_nodes, d_model = h_e.shape        if x_orig.dim() == 4:            x_reshape = x_orig.permute(1, 0, 2, 3)        elif x_orig.dim() == 3:            x_reshape = x_orig.permute(1, 0, 2).reshape(batch_size, seq_len, num_nodes, -1)        else:            raise ValueError(f"x_orig dim {x_orig.dim()} not supported")        wind_vars = x_reshape[:, :, :, 4:6]        last_wind = wind_vars[:, -1]        b, n, _ = last_wind.shape        wind_flat = last_wind.reshape(b, n, -1)        # 做了一个动态优化        # 从所有节点中，为每个地点 i 按“风驱动相似度/邻近性”选择 k 个最有可能将污染传输到 i 的邻居        # sim（相似度）越大，说明两个节点的风速/风向越接近，也越可能有：距离相近、污染传输相关        dist = torch.cdist(wind_flat, wind_flat)        sim = -dist        k = min(self.k_neighbors, n)        topk_idx = sim.topk(k=k, dim=-1).indices        t0 = seq_len - 1        t_start = max(0, t0 - self.tau + 1)        hist = h_e[:, t_start:t0 + 1].permute(0, 2, 1, 3)        tau_eff = hist.size(2)        batch_idx = torch.arange(b, device=h_e.device).view(b, 1, 1).expand(b, n, k)        neighbor_hist = hist[batch_idx, topk_idx]        neighbor_hist = neighbor_hist.reshape(b, n, k * tau_eff, d_model)        # local attention        q = h_e[:, t0]        q = self.q_proj(q).unsqueeze(2)        k_feat = self.k_proj(neighbor_hist)        v_feat = self.v_proj(neighbor_hist)        scale = math.sqrt(d_model)        attn_scores = (q * k_feat).sum(-1) / scale        attn_weights = torch.softmax(attn_scores, dim=-1).unsqueeze(-1)        context = (attn_weights * v_feat).sum(2)        h_l = self.mlp(context)        return h_lclass DiffeqSolver:    def __init__(self, method, odeint_rtol=1e-5,                 odeint_atol=1e-5, adjoint=True):        self.ode_method = method        self.odeint = odeint        self.rtol = odeint_rtol        self.atol = odeint_atol    def solve(self, odefunc, first_point, time_steps_to_pred):        pred_y = self.odeint(odefunc,                             first_point,                             time_steps_to_pred,                             rtol=self.rtol,                             atol=self.atol,                             method=self.ode_method)        return pred_yclass ODEFunc(nn.Module):    def __init__(self, gcn_hidden_dim, input_dim, adj_mx, edge_index, edge_attr,                 K_neighbour, num_nodes, device, num_layers=2,                 activation='tanh', filter_type="diff_adv", estimate=False):        super(ODEFunc, self).__init__()        self.device = device        self._activation = torch.tanh if activation == 'tanh' else torch.relu        self.num_nodes = num_nodes        self.gcn_hidden_dim = gcn_hidden_dim        self.input_dim = input_dim        self.num_layers = num_layers                self._filter_type = filter_type        self.adj_mx = adj_mx        self.edge_index = torch.tensor(edge_index, dtype=torch.int64).to(self.device)        self.edge_attr = edge_attr        self.K_neighbour = K_neighbour        self.diff_edge_attr = self.edge_attr[:, 0]        self.adv_edge_attr = None        self.source_sink = None        self.source_sink_pred = nn.Linear(128+128,64)        self.source_embed = nn.Linear(self.gcn_hidden_dim+1,1)        self.norm = nn.LayerNorm(self.num_nodes)        self.residual = nn.Identity()        self.diff_cheb_conv = self.laplacian_operator()        self.adv_cheb_conv = self.laplacian_operator()                # batch_size, node_num, hidden        self.previous_x = torch.randn(64,187,64).to(self.device)    def create_adv_matrix(self, last_wind_vars, wind_mean, wind_std):        batch_size = last_wind_vars.shape[0]        edge_src, edge_target = self.edge_index        node_src = last_wind_vars[:, edge_src, :]        node_target = last_wind_vars[:, edge_target, :]        src_wind_speed = node_src[:, :, 0] * wind_std[0] + wind_mean[0]    # km/h        src_wind_dir = node_src[:, :, 1] * wind_std[1] + wind_mean[1]        dist = self.edge_attr[:, 1].unsqueeze(dim=0).repeat(batch_size, 1)        dist_dir = self.edge_attr[:, 2].unsqueeze(dim=0).repeat(batch_size, 1)        src_wind_dir = (src_wind_dir + 180) % 360        theta = torch.abs(dist_dir - src_wind_dir)        adv_edge_attr = F.relu(3 * src_wind_speed * torch.cos(theta) / dist)  # B x M        return adv_edge_attr    def create_equation(self, last_wind_vars, wind_mean, wind_std):        self.adv_edge_attr = self.create_adv_matrix(last_wind_vars, wind_mean, wind_std)    def create_source_matrix(self, features):        source_term = self.source_sink_pred(features) # B,N,64        self.source_sink = source_term    def forward(self, t_local, Xt):        grad_diff = self.ode_func_net_diff(Xt, self.diff_edge_attr)        grad_adv = self.ode_func_net_adv(Xt, self.adv_edge_attr)        grad_source = self.ode_func_net_source_sink(Xt, self.source_sink)        # print(grad_diff.shape, grad_adv.shape, grad_source.shape)        grad = 0.1 * grad_diff + grad_adv + grad_source        return grad    def ode_func_net_source_sink(self, x, source):        out = torch.cat([x.unsqueeze(-1), source], dim=-1)        out = self.norm(self.source_embed(out).squeeze(-1))        return out    def ode_func_net_diff(self, x, edge_attr):        # x: B x N*var_dim        batch_size = x.shape[0]        x = torch.reshape(x, (batch_size, self.num_nodes, self.input_dim))        x = self.diff_cheb_conv[0](x, self.edge_index, edge_attr, lambda_max=2)        x = self._activation(x)        for op in self.diff_cheb_conv[1:-1]:            residual = self.residual(x)            x = op(x, self.edge_index, edge_attr, lambda_max=2)            x = self._activation(x) + residual        x = self.diff_cheb_conv[-1](x, self.edge_index, edge_attr, lambda_max=2)        return x.reshape((batch_size, self.num_nodes * self.input_dim))    def ode_func_net_adv(self, x, edge_attr):        batch_size = x.shape[0]        batch = torch.arange(0, batch_size)        batch = torch.repeat_interleave(batch, self.num_nodes).to(self.device)        x = x.reshape(batch_size * self.num_nodes, -1)  # B*N x input_dim        x = x + 0.01 * self.previous_x.sum(dim=1).sum(dim=-1).reshape(batch_size * self.num_nodes, -1)        edge_indices = []        for i in range(batch_size):            edge_indices.append(self.edge_index + i * self.num_nodes)        edge_index = torch.cat(edge_indices, dim=1)  # 2 x B*M        edge_attr = edge_attr.flatten()  # B*M        x = self.adv_cheb_conv[0](x, edge_index, edge_attr, batch=batch, lambda_max=2)        x = self._activation(x)        for op in self.adv_cheb_conv[1:-1]:            residual = self.residual(x)            x = op(x, edge_index, edge_attr, batch=batch, lambda_max=2)            x = self._activation(x) + residual        x = self.adv_cheb_conv[-1](x, edge_index, edge_attr, batch=batch, lambda_max=2)        x = x.reshape(batch_size, self.num_nodes, self.input_dim)        return x.reshape((batch_size, self.num_nodes * self.input_dim))    @staticmethod    def dense_to_sparse(adj: torch.Tensor):        batch_size, num_nodes, _ = adj.size()        edge_indices = []        edge_attrs = []        for i in range(batch_size):            edge_index, edge_attr = dense_to_sparse(adj[i])            edge_indices.append(edge_index + i * num_nodes)            edge_attrs.append(edge_attr)        edge_index = torch.cat(edge_indices, dim=1)        edge_attr = torch.cat(edge_attrs, dim=0)        return edge_index, edge_attr    def laplacian_operator(self):        # approximate Laplacian        operator = nn.ModuleList()        operator.append(            ChebConv(in_channels=self.input_dim, out_channels=self.gcn_hidden_dim,                     K=self.K_neighbour, normalization='sym',                     bias=True)        )        for _ in range(self.num_layers - 2):            operator.append(                ChebConv(in_channels=self.gcn_hidden_dim, out_channels=self.gcn_hidden_dim,                         K=self.K_neighbour, normalization='sym',bias=True)            )        operator.append(            ChebConv(in_channels=self.gcn_hidden_dim, out_channels=self.input_dim,                     K=self.K_neighbour, normalization='sym', bias=True)        )        return operatorclass Model(nn.Module):    def __init__(self, adj_mx, edge_index, edge_attr, node_attr, wind_mean,wind_std,                num_nodes, input_dim, output_dim, horizon, rnn_units, num_layers=1, cheb_k=3,                 ycov_dim=5, mem_num=20, mem_dim=64, cl_decay_steps=2000, use_curriculum_learning=True):        super(Model, self).__init__()        self.adj_mx = adj_mx        self.edge_index = torch.tensor(edge_index, dtype=torch.int32).to("cuda:0")        self.edge_attr = torch.from_numpy(edge_attr).float().to("cuda:0")        self.node_attr = node_attr        self.wind_mean = wind_mean        self.wind_std = wind_std        self.wind_mean = wind_mean[-2:]        self.wind_std = wind_std[-2:]        self.num_nodes = num_nodes        self.input_dim = input_dim        self.rnn_units = rnn_units        self.output_dim = output_dim        self.horizon = horizon        self.num_layers = num_layers        self.cheb_k = cheb_k        self.ycov_dim = ycov_dim        self.cl_decay_steps = cl_decay_steps        self.use_curriculum_learning = use_curriculum_learning                # memory        self.mem_num = mem_num        self.mem_dim = mem_dim        self.glo_memory = self.construct_global_memory()        self.loc_memory = LocalMemoryModule(num_nodes=self.num_nodes, d_model=self.rnn_units, tau=3, k_neighbors=8)        self.memory_embed = nn.Sequential(nn.Linear(self.mem_dim+self.mem_dim, self.mem_dim, bias=True))        # encoder        self.encoder = STEncoder(self.num_nodes, self.input_dim, self.rnn_units, self.cheb_k, self.num_layers)                # deocoder        self.decoder_dim = self.rnn_units + self.mem_dim        self.decoder = STDecoder(self.num_nodes, self.output_dim + self.ycov_dim, self.decoder_dim, self.cheb_k, self.num_layers)        # solver        self.phy_solver = DiffeqSolver(            method="dopri5",            odeint_atol=1e-2,            odeint_rtol=1e-2,            adjoint=True        )        self.phy_odefunc = ODEFunc(gcn_hidden_dim=64, input_dim=1, adj_mx=self.adj_mx, edge_index=self.edge_index, edge_attr=self.edge_attr,                 K_neighbour=3, num_nodes=184, device="cuda:0", num_layers=3,                 activation='tanh', filter_type="diff_adv", estimate=False)                self.phy_output = nn.Linear(self.decoder_dim, self.output_dim, bias=True)        self.y_cov_embed_layer = nn.Linear(24*5, self.decoder_dim)                # output        self.proj = nn.Sequential(nn.Linear(self.decoder_dim, self.output_dim, bias=True))        self.setting = self.get_setting()        def construct_global_memory(self):        memory_dict = nn.ParameterDict()        memory_dict['Memory'] = nn.Parameter(torch.randn(self.mem_num, self.mem_dim), requires_grad=True)     # (M, d)        memory_dict['Wq'] = nn.Parameter(torch.randn(self.rnn_units, self.mem_dim), requires_grad=True)        memory_dict['We1'] = nn.Parameter(torch.randn(self.num_nodes, self.mem_num), requires_grad=True)        memory_dict['We2'] = nn.Parameter(torch.randn(self.num_nodes, self.mem_num), requires_grad=True)        for param in memory_dict.values():            nn.init.xavier_normal_(param)        return memory_dict        def global_memory_modeling(self, h_t:torch.Tensor):        query = torch.matmul(h_t, self.glo_memory['Wq'])     # (B, N, d)        att_score = torch.softmax(torch.matmul(query, self.glo_memory['Memory'].t()), dim=-1)         # alpha: (B, N, M)        value = torch.matmul(att_score, self.glo_memory['Memory'])     # (B, N, d)        return value, query                def forward(self, x, y_cov, labels=None, batches_seen=None):        # inputs data        # x: T B N D        # y_cov: T_pred B N (D-1)        x_orig = x.clone() # T B N D        seq_len, batch_size = x.size(0), x.size(1)        x = x.reshape(seq_len, batch_size, self.num_nodes, self.input_dim).permute(1,0,2,3) # B T N D        # STencoder        node_embeddings1 = torch.matmul(self.glo_memory['We1'], self.glo_memory['Memory'])        node_embeddings2 = torch.matmul(self.glo_memory['We2'], self.glo_memory['Memory'])        g1 = F.softmax(F.relu(torch.mm(node_embeddings1, node_embeddings2.T)), dim=-1)        g2 = F.softmax(F.relu(torch.mm(node_embeddings2, node_embeddings1.T)), dim=-1)        supports = [g1, g2]        init_state = self.encoder.init_hidden(x.shape[0])        h_en, _ = self.encoder(x, init_state, supports) # B, T, N, hidden        h_t = h_en[:, -1, :, :] # B, N, hidden (last state)                        # global memory        h_global, query = self.global_memory_modeling(h_t)                # local memory        h_local = self.loc_memory(h_en, x_orig)        # print(h_global.shape, h_local.shape)        h_memory = self.memory_embed(torch.cat([h_global, h_local], dim=-1)) # B, N, hidden        h_embed = torch.cat([h_t,h_memory], dim=-1) # B, N, hidden+hidden        # h_embed = torch.cat([h_global, h_local], dim=-1)        # print(h_embed.shape)        ht_list = [h_embed]*self.num_layers                # func init        x_reshape = x_orig.reshape(seq_len, batch_size, self.num_nodes, -1)        wind_vars = x_reshape[:, :, :, 4: 6]  # T x B x N x 2   wind speed and wind direction        last_wind_vars = wind_vars[-1]  # B x N x 2        self.phy_odefunc.create_equation(last_wind_vars, self.wind_mean, self.wind_std)                # func inputs        tau_back = 3  # 回溯窗口\tau 是3        self.phy_odefunc.previous_x = h_en[:,-1-tau_back:-1,:,:]                phy_input = x_reshape[-1,:,:,0].reshape(batch_size,-1)        y_cov_embed = self.y_cov_embed_layer(y_cov.permute(1,2,3,0).reshape(batch_size, self.num_nodes, -1)).squeeze(-1)        self.phy_odefunc.create_source_matrix(torch.cat([h_embed, y_cov_embed], dim=-1))        time_steps_to_predict = torch.arange(start=0, end=self.horizon + 1, step=1).float()  # horizon 1 + 24        time_steps_to_predict = time_steps_to_predict / len(time_steps_to_predict)                # future evo        phy_y = self.phy_solver.solve(self.phy_odefunc, phy_input, time_steps_to_predict)  # T x B x N*D        phy_y = phy_y[1:]        # STDecoder        out = []        for t in range(self.horizon):            h_de, ht_list = self.decoder(torch.cat([phy_y[t, ...].unsqueeze(-1), y_cov[t, ...]], dim=-1), ht_list, supports)            h_de = self.proj(h_de)            out.append(h_de)        output = torch.stack(out, dim=1)        output = output.squeeze(-1)        output = output.permute(1,0,2)        return output        def get_setting(self):        setting = 'airdde'        return settingimport osimport warningsimport numpy as npimport timeimport torchimport torch.nn as nnfrom torch import optimfrom torch.utils.tensorboard import SummaryWriterfrom data_loader import data_providerfrom model import Modelfrom utils import *warnings.filterwarnings('ignore')class Exp_Basic(object):    def __init__(self, args):        self.args = args        self.model_dict = {            "AirDDE": Model,        }        self.device = self._acquire_device(args.GPU)        self.model = self._build_model().to(self.device)    def _build_model(self):        raise NotImplementedError    def _build_TB_logger(self, setting):        log_dir = os.path.join(self.args.TB_dir, setting)        if not os.path.exists(log_dir):            os.makedirs(log_dir)        logger = SummaryWriter(log_dir)        return logger    def _acquire_device(self, args):        if args.use_gpu:            device = torch.device('cuda:{}'.format(args.gpu))            print('Use GPU: cuda:{}'.format(args.gpu))        else:            device = torch.device('cpu')            print('Use CPU')        return device    def _get_data(self, **kwargs):        pass    def vali(self, **kwargs):        pass    def train(self, **kwargs):        pass    def test(self, **kwargs):        passclass Exp_Air(Exp_Basic):    def __init__(self, args):        adj_mx, edge_index, edge_attr, node_attr = load_graph_data(args.data.root_path)        args.adj_mx = adj_mx    # N x N        args.edge_index = edge_index    # adjacent list: 2 x M        args.edge_attr = edge_attr      # M x D        args.node_attr = node_attr      # N x D        if args.to_log_file:            self._log_dir = self._get_log_dir(args)        else:            self._log_dir = None        self._logger = get_logger(self._log_dir, args.model_name, 'info.log',                                  level=args.log_level, to_stdout=args.to_stdout)        args.logger = self._logger        if args.data.embed:            args.model.input_dim = int(args.model.input_dim) + int(args.model.embed_dim)        super(Exp_Air, self).__init__(args)        self.num_nodes = adj_mx.shape[0]        self.input_var = int(self.args.model.input_dim)        self.input_dim = int(self.args.model.X_dim)        self.seq_len = int(self.args.model.seq_len)        self.horizon = int(self.args.model.horizon)        self.output_dim = int(self.args.model.X_dim)    # def prepare_x_y(x, y):    #     """    #     :param x: shape (batch_size, seq_len, num_sensor, input_dim)    #     :param y: shape (batch_size, horizon, num_sensor, input_dim)    #     :return1: x shape (seq_len, batch_size, num_sensor, input_dim)    #             y shape (horizon, batch_size, num_sensor, input_dim)    #     :return2: x: shape (seq_len, batch_size, num_sensor * input_dim)    #             y: shape (horizon, batch_size, num_sensor * output_dim)    #     """    #     x0 = x[..., :args.input_dim]    #     y0 = y[..., :args.output_dim]    #     y1 = y[..., args.output_dim:]    #     x0 = torch.from_numpy(x0).float()    #     y0 = torch.from_numpy(y0).float()    #     y1 = torch.from_numpy(y1).float()    #     return x0.to(device), y0.to(device), y1.to(device) # x, y, y_cov    def _build_model(self):        dataset, _ = self._get_data('val')        self.args.data.mean_ = dataset.scaler.mean_        self.args.data.std_ = dataset.scaler.scale_        model = self.model_dict[self.args.model_name](            adj_mx = self.args.adj_mx,             edge_index = self.args.edge_index,             edge_attr = self.args.edge_attr,             node_attr = self.args.node_attr,            wind_mean = self.args.data.mean_,            wind_std = self.args.data.std_,            num_nodes=int(self.args.num_nodes),            input_dim=int(self.args.input_dim),            output_dim=int(self.args.output_dim),            horizon=int(self.args.horizon),            rnn_units=int(self.args.rnn_units),            num_layers=int(self.args.num_rnn_layers),             cheb_k=2,            ycov_dim=5,             mem_num=20,             mem_dim=64,             cl_decay_steps=2000,             use_curriculum_learning=True        ).float()        self._logger.info("Model created")        self._logger.info(            "Total trainable parameters {}".format(count_parameters(model))        )        if self.args.GPU.use_multi_gpu and self.args.GPU.use_gpu:            model = nn.DataParallel(model, device_ids=self.args.GPU.device_ids)        return model    def _get_data(self, flag):        data_set, data_loader = data_provider(self.args, flag)        return data_set, data_loader    def _select_optimizer(self):        model_optim = optim.Adam(self.model.parameters(), lr=self.args.train.lr, eps=1e-8)        return model_optim    def _select_criterion(self):        if self.args.model.loss.criterion == "mse":            criterion = nn.MSELoss()        elif self.args.model.loss.criterion == "mae":            criterion = nn.L1Loss()        else:            criterion = nn.L1Loss()        return criterion    def _select_lr_scheduler(self, optimizer, train_loader):        if self.args.train.lradj == 'MultiStep':            lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=self.args.train.steps,                                                                gamma=self.args.train.lr_decay_ratio)        elif self.args.train.lradj == 'TST':            lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer,                                                               steps_per_epoch=len(train_loader),                                                               pct_start=self.args.train.pct_start,                                                               epochs=self.args.train.epochs,                                                               max_lr=self.args.train.lr)        else:            lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)        return lr_scheduler    def vali(self, vali_data, vali_loader, epoch_num, save=False):        with torch.no_grad():            self.model.eval()            preds = []            truths = []            batches_seen = 0            for i, (x, gt) in enumerate(vali_loader):                x, gt, y_embed = self._prepare_data(x, gt)                output = self.model(x, y_embed, gt,batches_seen)                batches_seen += 1                truths.append(gt.cpu())    # T x B x N                preds.append(output.cpu())                loss = self.pred_loss(gt, output)                                if self.TB_logger:                    self.TB_logger.add_scalar("val/loss", loss, epoch_num * len(vali_loader) + i)            truths = torch.cat(truths, dim=1)            preds = torch.cat(preds, dim=1)   # T x B x N            val_loss = self.criterion(truths, preds)            truths = truths.permute(1, 0, 2)            preds = preds.permute(1, 0, 2)   # B x T x N            mae, smape, rmse = self._compute_loss_eval(truths, preds)            self._logger.info('Evaluation: - mae - {:.4f} - smape - {:.4f} - rmse - {:.4f}'                              .format(mae, smape, rmse))            return val_loss    def train(self):        if self.args.TB_dir:            self.TB_logger = self._build_TB_logger(self.model.setting)        else:            self.TB_logger = None        self._logger.info('Model mode: train')        train_data, train_loader = self._get_data(flag='train')        vali_data, vali_loader = self._get_data(flag='val')        self.inverse_transform = train_data.inverse_transform        self.criterion = self._select_criterion()        model_save_path = os.path.join(self.args.checkpoints, self.model.setting)        if not os.path.exists(model_save_path):            os.makedirs(model_save_path)        optimizer = self._select_optimizer()        early_stopping = EarlyStopping(patience=self.args.train.patience, verbose=True, logger=self._logger)        lr_scheduler = self._select_lr_scheduler(optimizer, train_loader)        time_now = time.time()        train_steps = len(train_loader)        self._logger.info('Start training ...')        num_batches = self.args.data.batch_size        self._logger.info("num_batches: {}".format(num_batches))        batches_seen = 0        for epoch_num in range(1, self.args.train.epochs + 1):            if self.args.to_stdout:                print('\nTrain epoch %s:' % (epoch_num))            self.model.train()            losses = []            iter_count = 0            for i, (batch_x, batch_y) in enumerate(train_loader):                iter_count += 1                optimizer.zero_grad()                batch_x, batch_y, y_embed = self._prepare_data(batch_x, batch_y)                output = self.model(batch_x, y_embed, batch_y,batches_seen)                loss = self.pred_loss(batch_y, output)                self._logger.debug(loss.item())                losses.append(loss.item())                                batches_seen += 1                loss.backward()                optimizer.step()                if self.TB_logger:                    self.TB_logger.add_scalar("train/loss", loss.item(), epoch_num * train_steps + i)                if self.args.train.lradj == 'TST':                    lr_scheduler.step()                del output, loss, batch_x, batch_y                torch.cuda.empty_cache()            val_loss = self.vali(vali_data, vali_loader, epoch_num)            if (epoch_num % self.args.train.log_every) == self.args.train.log_every - 1:                speed = (time.time() - time_now) / iter_count                left_time = speed * ((self.args.train.epochs - epoch_num) * train_steps - i)                message = ('Epoch [{}/{}] train_loss: {:.4f}, val_loss: {:.4f}, lr: {:.6f}'                           .format(epoch_num, self.args.train.epochs,                                   np.mean(losses), val_loss, optimizer.param_groups[0]['lr']))                self._logger.info(message)                self._logger.info('speed: {:.4f}s/iter; left time: {:.4f}s'.format(speed, left_time))                iter_count = 0                time_now = time.time()            # 学习率动态调整            if self.args.train.lradj == 'MultiStep':                lr_scheduler.step()            elif self.args.train.lradj == 'TST':                pass            else:                lr_scheduler.step(val_loss)            early_stopping(val_loss, self.model, model_save_path)            if early_stopping.early_stop:                print("Early stopping")                break            self._logger.info("---" * 30)    @staticmethod    def _get_log_dir(args):        log_dir = args.train.get('log_dir')        if log_dir is None:            run_id = '%s_%s/' % (                args.model_name, time.strftime('%m-%d-%H-%M-%S'))            base_dir = args.log_base_dir            log_dir = os.path.join(base_dir, run_id)        if not os.path.exists(log_dir):            os.makedirs(log_dir)        return log_dir    def _prepare_data(self, x, y):        x, y = self._get_x_y(x, y)  # B x 24(72 hours) x N x D        x, y, y_embed = self._get_x_y_in_correct_dims(x, y)  # 24 x B x N x D        return x.to(self.device), y.to(self.device), y_embed.to(self.device)  # 24 x B x 35 * 11    def _get_x_y(self, x, y):        x = x.float()        y = y.float()        x = x.permute(1, 0, 2, 3)        y = y.permute(1, 0, 2, 3)        return x, y    def _get_x_y_in_correct_dims(self, x, y):        # print(2333)        # print(x.shape)        # print(y.shape)        batch_size = x.size(1)        if self.args.data.embed:            station_x = torch.arange(0, self.num_nodes).unsqueeze(0).unsqueeze(0).unsqueeze(-1).repeat(self.seq_len, batch_size, 1, 1)            station_y = torch.arange(0, self.num_nodes).unsqueeze(0).unsqueeze(0).unsqueeze(-1).repeat(self.horizon, batch_size, 1, 1)            x = torch.cat([x, station_x], dim=-1)            y = torch.cat([y, station_y], dim=-1)            x = x.reshape(self.seq_len, batch_size, self.num_nodes * self.input_var)            embed = [6, 7, 8, 9, 10, 11]            y_embed = y[..., embed].reshape(self.horizon, batch_size, self.num_nodes*len(embed))            y = y[..., :self.output_dim].reshape(self.horizon, batch_size,                                              self.num_nodes*self.output_dim)        else:            x0 = x[..., :self.input_var].reshape(self.seq_len, batch_size, self.num_nodes * self.input_var)            y0 = y[..., :self.output_dim].reshape(self.horizon, batch_size,                                                 self.num_nodes * self.output_dim)            y_embed = y[..., self.output_dim:].reshape(self.horizon, batch_size, self.num_nodes, -1)        return x0, y0, y_embed    def pred_loss(self, y_true, y_predicted):        y_true = self.inverse_transform(y_true)        y_predicted = self.inverse_transform(y_predicted)        return masked_loss(y_predicted, y_true, MAE)    def _compute_loss_eval(self, y_true, y_predicted):        y_true = self.inverse_transform(y_true)        y_predicted = self.inverse_transform(y_predicted)        return compute_all_metrics(y_predicted, y_true)    def kl_loss(self, mu, logvar):        # n_traj x B x N x Latent_dim        var = torch.exp(logvar)        loss = 1/2 * (var + mu**2 - logvar - 1)        return torch.mean(loss.sum(dim=(-1, -2)))    def get_loss(self, x_true, y_true):        loss = torch.zeros(1).to(self.device)        x_true = torch.reshape(x_true, (self.seq_len, -1, self.num_nodes, self.input_dim))        x_true = x_true[:, :, :, 0]        if self.args.model.loss.kl_loss:            kl_loss = self.kl_loss(self.model.means_z0, self.model.logvar_z0)            loss += kl_loss        if self.args.model.loss.recon_loss:            recon_loss = self.criterion(x_true, self.model.recon_x)            loss += self.args.model.loss.recon_coeff * recon_loss        if self.args.model.loss.pred_loss:            pred_loss = self.criterion(y_true, self.model.pred_y)            loss += pred_loss        if self.args.model.loss.cl_loss:            loss += self.args.model.loss.cl_coeff * self.model.loss_CL        return lossimport sysimport yamlimport osgpu_list = "0"device_map = {gpu: i for i, gpu in enumerate(gpu_list.split(','))}os.environ["CUDA_VISIBLE_DEVICES"] = gpu_listsys.path.append('../AirDDE')import argparseimport torchimport randomfrom utils import parsing_syntax, ConfigDict, load_config, update_config, fix_seedfrom trainer import Exp_Airif True:    parser = argparse.ArgumentParser(description='AirDDE')    parser.add_argument('--config_filename', type=str, default='./configs/knowair_config.yaml', help='Configuration yaml file')    parser.add_argument('--itr', type=int, default=1, help='Number of experiments.')    parser.add_argument('--random_seed', type=int, default=2024, help='Random seed.')    parser.add_argument('--des', type=str, default='1', help="description of experiment.")    parser.add_argument('--num_nodes', type=int, default=184, help='num_nodes')    parser.add_argument('--seq_len', type=int, default=24, help='input sequence length')    parser.add_argument('--horizon', type=int, default=24, help='output sequence length')    parser.add_argument('--input_dim', type=int, default=6, help='number of input channel')    parser.add_argument('--output_dim', type=int, default=1, help='number of output channel')    parser.add_argument('--max_diffusion_step', type=int, default=3, help='max diffusion step or Cheb K')    parser.add_argument('--num_rnn_layers', type=int, default=1, help='number of rnn layers')    parser.add_argument('--rnn_units', type=int, default=64, help='number of rnn units')    parser.add_argument('--mem_num', type=int, default=20, help='number of meta-nodes/prototypes')    parser.add_argument('--mem_dim', type=int, default=64, help='dimension of meta-nodes/prototypes')    parser.add_argument("--loss", type=str, default='mask_mae_loss', help="mask_mae_loss")    parser.add_argument('--lamb', type=float, default=0.01, help='lamb value for separate loss')    parser.add_argument('--lamb1', type=float, default=0.01, help='lamb1 value for compact loss')    parser.add_argument("--epochs", type=int, default=200, help="number of epochs of training")    parser.add_argument("--patience", type=int, default=20, help="patience used for early stop")    parser.add_argument("--batch_size", type=int, default=64, help="size of the batches")    parser.add_argument("--lr", type=float, default=0.01, help="base learning rate")    parser.add_argument("--steps", type=eval, default=[50, 100], help="steps")    parser.add_argument("--lr_decay_ratio", type=float, default=0.1, help="lr_decay_ratio")    parser.add_argument("--epsilon", type=float, default=1e-3, help="optimizer epsilon")    parser.add_argument("--max_grad_norm", type=int, default=5, help="max_grad_norm")    parser.add_argument("--use_curriculum_learning", type=eval, choices=[True, False], default='True', help="use_curriculum_learning")    parser.add_argument("--cl_decay_steps", type=int, default=2000, help="cl_decay_steps")    parser.add_argument('--test_every_n_epochs', type=int, default=5, help='test_every_n_epochs')    parser.add_argument('--gpu', type=int, default=0, help='which gpu to use')    args, unknown = parser.parse_known_args()    unknown = parsing_syntax(unknown)    config = load_config(args.config_filename)    config = ConfigDict(config)    config = update_config(config, unknown)    for attr, value in config.items():        setattr(args, attr, value)    # random seed    fix_seed(args.random_seed)    args.GPU.use_gpu = True if torch.cuda.is_available() and args.GPU.use_gpu else False    if args.GPU.use_gpu and not args.GPU.use_multi_gpu:        try:            args.GPU.gpu = device_map[str(args.GPU.gpu)]        except KeyError:            raise KeyError("This GPU isn't available.")    if args.GPU.use_gpu and args.GPU.use_multi_gpu:        args.GPU.devices = args.GPU.devices.replace(' ', '')        device_ids = args.GPU.devices.split(',')        args.GPU.device_ids = [int(id_) for id_ in device_ids]        args.GPU.gpu = args.GPU.device_ids[0]    rmse_list, mae_list, mape_list = [], [], []    for exp_idx in range(args.itr):        args.exp_idx = exp_idx        if args.to_stdout:            print('\nNo%d experiment ~~~' % exp_idx)        exp = Exp_Air(args)        exp.train()        torch.cuda.empty_cache()